# 🧠 Autoregressive Language Models: RNN vs LSTM vs GRU
### Tutorial — NLP/Deep Learning Lab | Monday, 23rd March 2026
**Prepared by:** Suraj & Debajyoti | **Reviewed by:** Course Instructors  
**Theory Reference:** [Jurafsky & Martin, SLP3 Chapter 13](https://web.stanford.edu/~jurafsky/slp3/13.pdf)

---

### 📋 Notebook Outline
1. Setup & Imports
2. Theory Recap — RNN, LSTM, GRU (equations from SLP3 Ch.13)
3. Data Loading — WikiText-2 (Wikipedia corpus)
4. Model Implementations — Vanilla RNN, LSTM, GRU in PyTorch
5. Training — Shared loop with perplexity tracking
6. Quantitative Evaluation — Perplexity & BLEU comparison
7. Text Generation — Autoregressive sampling
8. Discussion — Architecture tradeoffs

> ⚡ **Tip:** `Runtime → Change runtime type → GPU (T4)` for 5× faster training!

## Section 1: Setup & Imports

In [ ]:
# Install packages (run once on Colab)
!pip install -q torch torchtext datasets sacrebleu tqdm pandas matplotlib

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import time, math, random, json
from collections import Counter
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


Using device: cuda
  GPU: NVIDIA A100 80GB PCIe


## Section 2: Theory Recap (SLP3 Chapter 13)

### 2.1 Autoregressive Language Modeling
A language model assigns probability to a word sequence via the chain rule:
$$P(w_1, ..., w_n) = \prod_{t=1}^{n} P(w_t \mid w_1, ..., w_{t-1})$$

In **autoregressive generation** (SLP3 §13.3.3): sample one word at a time, feed each sampled word back as input for the next step.

---
### 2.2 Vanilla RNN — Elman Network (SLP3 §13.1)

$$h_t = \tanh(U \cdot h_{t-1} + W \cdot e_t) \quad \text{[Eq. 13.5]}$$
$$\hat{y}_t = \text{softmax}(E^\top \cdot h_t) \quad \text{[weight tying, Eq. 13.14]}$$

- $e_t = E \cdot x_t$: word embedding for token at time $t$
- $U \in \mathbb{R}^{d_h \times d_h}$: recurrent weight matrix
- **Weight tying** (SLP3 §13.2.3): reuse embedding matrix $E$ at output, reducing parameters

**Problem:** Gradients decay exponentially over long sequences (vanishing gradient).

---
### 2.3 LSTM — Long Short-Term Memory (SLP3 §13.6)

| Gate | Equation | Role |
|------|----------|------|
| Forget | $f_t = \sigma(U_f h_{t-1} + W_f x_t)$ | What to erase from cell state |
| Input  | $i_t = \sigma(U_i h_{t-1} + W_i x_t)$ | What new info to write |
| Output | $o_t = \sigma(U_o h_{t-1} + W_o x_t)$ | What to expose as $h_t$ |

$$\tilde{c}_t = \tanh(U_g h_{t-1} + W_g x_t), \quad c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t, \quad h_t = o_t \odot \tanh(c_t)$$

**Key insight:** When $f_t \approx 1$, cell state $c_t \approx c_{t-1}$ — information flows unchanged across many steps, solving vanishing gradients.

---
### 2.4 GRU — Gated Recurrent Unit (Cho et al., 2014)

$$z_t = \sigma(U_z h_{t-1} + W_z x_t) \quad \text{[update gate]}$$
$$r_t = \sigma(U_r h_{t-1} + W_r x_t) \quad \text{[reset gate]}$$
$$\tilde{h}_t = \tanh(U(r_t \odot h_{t-1}) + W x_t), \quad h_t = (1-z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

GRU merges LSTM's forget+input gates into one update gate $z_t$, with **~25% fewer parameters** than LSTM.

---
### 2.5 Training: Teacher Forcing & Cross-Entropy Loss (SLP3 §13.2.2)

**Teacher forcing**: always feed the *true* previous word as input (not the model's prediction).  
**Cross-entropy loss** at each step: $\mathcal{L}_{CE} = -\log \hat{y}_t[w_{t+1}]$  
**Perplexity**: $\text{PPL} = \exp\left(\frac{1}{N}\sum_{t} \mathcal{L}_{CE}^{(t)}\right)$ — lower is better.

## Section 3: Data Loading — WikiText-2

**WikiText-2** (Merity et al., 2017) is a small but representative Wikipedia corpus:
- ~2.08M training tokens from verified, featured Wikipedia articles
- Clean, well-formed English (encyclopedic style)
- Standard LM benchmark — ideal for comparing architectures

In [2]:
from datasets import load_dataset

print("Loading WikiText-2...")
raw_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# Dataset statistics
for split in raw_dataset:
    texts = [t for t in raw_dataset[split]['text'] if t.strip()]
    tokens = sum(len(t.split()) for t in texts)
    print(f"  {split:12s}: {len(texts):6,} articles | {tokens:8,} tokens")

print()
# Sample line
samples = [t for t in raw_dataset['train']['text'] if len(t.strip()) > 100][:2]
for s in samples:
    print(f"  Sample: {s[:130]}...")
    print()


Loading WikiText-2...


Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 332103.17 examples/s]


  test        :  2,891 articles |  241,211 tokens
  train       : 23,767 articles | 2,051,910 tokens
  validation  :  2,461 articles |  213,886 tokens

  Sample:  Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred ...

  Sample:  The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained...



In [3]:
# ── Vocabulary Building ──────────────────────────────────────────────────────

def build_vocab(texts, min_freq=2, max_vocab=20000):
    """Word-level vocabulary with special tokens."""
    counter = Counter()
    for text in texts:
        counter.update(text.lower().split())
    vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
    for word, freq in counter.most_common(max_vocab):
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)
    return vocab

train_texts = [t for t in raw_dataset['train']['text'] if t.strip()]
vocab = build_vocab(train_texts)
idx2word = {v: k for k, v in vocab.items()}
VOCAB_SIZE = len(vocab)
print(f"Vocabulary size: {VOCAB_SIZE:,}")
print(f"Top-10 words: {list(vocab.keys())[4:14]}")


Vocabulary size: 20,004
Top-10 words: ['the', ',', '.', 'of', 'and', 'in', 'to', 'a', '=', '"']


In [4]:
# ── Dataset & DataLoader ─────────────────────────────────────────────────────

class WikiTextDataset(Dataset):
    """
    Sliding-window sequences for language modeling.
    Each sample: (x, y) where y = x shifted right by 1 (teacher forcing, SLP3 §13.2.2).
    """
    def __init__(self, texts, vocab, seq_len=64):
        self.seq_len = seq_len
        # Tokenise & concatenate all articles with <eos> separator
        all_tokens = []
        for text in texts:
            if text.strip():
                tokens = [vocab.get(w, vocab['<unk>']) for w in text.lower().split()]
                if tokens:
                    all_tokens.extend(tokens)
                    all_tokens.append(vocab['<eos>'])
        self.tokens = torch.tensor(all_tokens, dtype=torch.long)
        self.n_seqs = (len(self.tokens) - 1) // seq_len

    def __len__(self): return self.n_seqs

    def __getitem__(self, idx):
        s = idx * self.seq_len
        return self.tokens[s:s+self.seq_len], self.tokens[s+1:s+self.seq_len+1]

SEQ_LEN, BATCH_SIZE = 64, 64

print("Building datasets...")
train_ds = WikiTextDataset([t for t in raw_dataset['train']['text'] if t.strip()], vocab, SEQ_LEN)
val_ds   = WikiTextDataset([t for t in raw_dataset['validation']['text'] if t.strip()], vocab, SEQ_LEN)
test_ds  = WikiTextDataset([t for t in raw_dataset['test']['text'] if t.strip()], vocab, SEQ_LEN)

print(f"  Train: {len(train_ds):,} sequences")
print(f"  Val:   {len(val_ds):,} sequences")
print(f"  Test:  {len(test_ds):,} sequences")

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  pin_memory=True, num_workers=2)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, pin_memory=True)
print(f"\nLoaders ready. Batches — Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")


Building datasets...
  Train: 32,432 sequences
  Val:   3,380 sequences
  Test:  3,814 sequences

Loaders ready. Batches — Train: 507, Val: 53, Test: 60


## Section 4: Model Implementations in PyTorch

In [5]:
# ── 4.1 Vanilla RNN (SLP3 §13.1, §13.2) ────────────────────────────────────

class RNNLanguageModel(nn.Module):
    """
    Elman RNN Language Model (SLP3 §13.1).
    hₜ = tanh(U·hₜ₋₁ + W·eₜ)        [Eq 13.5]
    ŷₜ = softmax(Eᵀ·hₜ)               [weight-tied output, Eq 13.14]
    """
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_size, self.num_layers = hidden_size, num_layers
        self.model_type = 'RNN'
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.rnn = nn.RNN(embed_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers>1 else 0, nonlinearity='tanh')
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size, bias=False)
        self.fc.weight = self.embedding.weight          # ← WEIGHT TYING (SLP3 §13.2.3)
        self._init_weights()

    def _init_weights(self):
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        for n, p in self.rnn.named_parameters():
            nn.init.orthogonal_(p) if 'weight' in n else nn.init.zeros_(p)

    def forward(self, x, hidden=None):
        out, hidden = self.rnn(self.drop(self.embedding(x)), hidden)
        return self.fc(self.drop(out)), hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)

print("✅ RNNLanguageModel defined")


✅ RNNLanguageModel defined


In [6]:
# ── 4.2 LSTM (SLP3 §13.6) ────────────────────────────────────────────────────

class LSTMLanguageModel(nn.Module):
    """
    LSTM Language Model (SLP3 §13.6).
    Gates: forget fₜ, input iₜ, output oₜ
    Cell: cₜ = fₜ⊙cₜ₋₁ + iₜ⊙c̃ₜ   [protected memory highway]
    Hidden: hₜ = oₜ⊙tanh(cₜ)
    """
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_size, self.num_layers = hidden_size, num_layers
        self.model_type = 'LSTM'
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers>1 else 0)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size, bias=False)
        self.fc.weight = self.embedding.weight          # ← WEIGHT TYING
        self._init_weights()

    def _init_weights(self):
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        for n, p in self.lstm.named_parameters():
            if 'weight_ih' in n: nn.init.xavier_uniform_(p)
            elif 'weight_hh' in n: nn.init.orthogonal_(p)
            elif 'bias' in n:
                nn.init.zeros_(p)
                # Forget-gate bias = 1: helps long-range memory (Jozefowicz et al. 2015)
                p.data[p.size(0)//4 : p.size(0)//2].fill_(1.0)

    def forward(self, x, hidden=None):
        out, hidden = self.lstm(self.drop(self.embedding(x)), hidden)
        return self.fc(self.drop(out)), hidden

    def init_hidden(self, batch_size):
        z = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)
        return (z, z.clone())

print("✅ LSTMLanguageModel defined")


✅ LSTMLanguageModel defined


In [7]:
# ── 4.3 GRU (Cho et al., 2014) ───────────────────────────────────────────────

class GRULanguageModel(nn.Module):
    """
    GRU Language Model (Cho et al., 2014).
    Gates: update zₜ, reset rₜ  (no separate cell state — simpler than LSTM)
    hₜ = (1-zₜ)⊙hₜ₋₁ + zₜ⊙h̃ₜ   [interpolate old and new]
    ~25% fewer parameters than LSTM for same hidden_size.
    """
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_size, self.num_layers = hidden_size, num_layers
        self.model_type = 'GRU'
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers>1 else 0)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size, bias=False)
        self.fc.weight = self.embedding.weight          # ← WEIGHT TYING
        self._init_weights()

    def _init_weights(self):
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        for n, p in self.gru.named_parameters():
            nn.init.orthogonal_(p) if 'weight' in n else nn.init.zeros_(p)

    def forward(self, x, hidden=None):
        out, hidden = self.gru(self.drop(self.embedding(x)), hidden)
        return self.fc(self.drop(out)), hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)

print("✅ GRULanguageModel defined")


✅ GRULanguageModel defined


In [8]:
# ── 4.4 Parameter Count & Model Sizes ────────────────────────────────────────

EMBED_SIZE  = 256
HIDDEN_SIZE = 256
NUM_LAYERS  = 2
DROPOUT     = 0.3

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

rnn_model  = RNNLanguageModel( VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)
lstm_model = LSTMLanguageModel(VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)
gru_model  = GRULanguageModel( VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)

print(f"{'='*58}")
print(f"  {'Model':<10} {'Params':>10}  {'Recurrent core':<25} Layers")
print(f"{'='*58}")
for m, n in [(rnn_model,'RNN'), (lstm_model,'LSTM'), (gru_model,'GRU')]:
    core = {'RNN':'1 gate (tanh)','LSTM':'3 gates + cell state','GRU':'2 gates (no cell)'}[n]
    print(f"  {n:<10} {count_params(m):>10,}  {core:<25} {NUM_LAYERS}")
print(f"{'='*58}")
print(f"\n  embed_size={EMBED_SIZE}, hidden_size={HIDDEN_SIZE}, dropout={DROPOUT}")
print(f"  Note: embedding ({VOCAB_SIZE}×{EMBED_SIZE}) weight-tied to output projection.")


  Model          Params  Recurrent core            Layers
  RNN         5,384,192  1 gate (tanh)             2
  LSTM        6,173,696  3 gates + cell state      2
  GRU         5,910,528  2 gates (no cell)         2

  embed_size=256, hidden_size=256, dropout=0.3
  Note: embedding (20004×256) weight-tied to output projection.


## Section 5: Training

**Key implementation details:**
- **Cross-Entropy Loss** (SLP3 Eq 13.10–13.11) — ignores `<pad>` tokens
- **Teacher forcing** — true token always fed as next input (SLP3 §13.2.2)
- **Gradient clipping** — prevents exploding gradients (critical for RNNs!)
- **Truncated BPTT** — hidden state detached between batches to limit backprop depth
- **Adam optimizer** + ReduceLROnPlateau scheduler

In [9]:
def compute_perplexity(loss):
    return math.exp(min(loss, 20))   # cap to prevent overflow

def detach_hidden(h):
    """Detach hidden state from computation graph (truncated BPTT)."""
    return tuple(x.detach() for x in h) if isinstance(h, tuple) else h.detach()

def train_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    total_loss, hidden = 0, None
    for x, y in tqdm(loader, desc="  train", leave=False):
        x, y = x.to(device), y.to(device)
        bs = x.size(0)
        if hidden is None or (isinstance(hidden,tuple) and hidden[0].size(1)!=bs) or            (not isinstance(hidden,tuple) and hidden.size(1)!=bs):
            hidden = model.init_hidden(bs)
        hidden = detach_hidden(hidden)
        optimizer.zero_grad()
        logits, hidden = model(x, hidden)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)   # ← GRADIENT CLIPPING
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, hidden = 0, None
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            bs = x.size(0)
            if hidden is None or (isinstance(hidden,tuple) and hidden[0].size(1)!=bs) or                (not isinstance(hidden,tuple) and hidden.size(1)!=bs):
                hidden = model.init_hidden(bs)
            hidden = detach_hidden(hidden)
            logits, hidden = model(x, hidden)
            total_loss += criterion(logits.view(-1, logits.size(-1)), y.view(-1)).item()
    return total_loss / len(loader)

print("✅ Training utilities defined")


✅ Training utilities defined


In [ ]:
def train_model(model, name, n_epochs=3, lr=3e-3):
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=1, factor=0.5)
    history = {'train_ppl': [], 'val_ppl': []}
    best_val_loss, best_state = float('inf'), None

    print(f"\n{'='*62}")
    print(f"  Training {name} | {count_params(model):,} params | {n_epochs} epochs")
    print(f"{'='*62}")

    for epoch in range(1, n_epochs+1):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_loss   = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)
        train_ppl, val_ppl = compute_perplexity(train_loss), compute_perplexity(val_loss)
        history['train_ppl'].append(train_ppl)
        history['val_ppl'].append(val_ppl)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  Epoch {epoch}/{n_epochs} | Train PPL: {train_ppl:7.1f} | Val PPL: {val_ppl:7.1f} | {time.time()-t0:.0f}s")

    model.load_state_dict(best_state)
    print(f"  ✅ Best val PPL: {compute_perplexity(best_val_loss):.1f}")
    return history

# ── Train all three models ────────────────────────────────────────────────────
N_EPOCHS = 3  # Increase to 5 for better results (≈10 min on T4)

rnn_hist  = train_model(rnn_model,  'Vanilla RNN', N_EPOCHS)
lstm_hist = train_model(lstm_model, 'LSTM',        N_EPOCHS)
gru_hist  = train_model(gru_model,  'GRU',         N_EPOCHS)



  Training Vanilla RNN | 5,384,192 params | 3 epochs


  Epoch 1/3 | Train PPL:   569.4 | Val PPL:   248.8 | 14s


  Epoch 2/3 | Train PPL:   275.1 | Val PPL:   203.1 | 13s


  Epoch 3/3 | Train PPL:   228.9 | Val PPL:   189.6 | 14s
  ✅ Best val PPL: 189.6

  Training LSTM | 6,173,696 params | 3 epochs


  train:  42%|████▏     | 215/507 [00:06<00:08, 34.83it/s]

## Section 6: Quantitative Evaluation

In [ ]:
# ── Test Perplexity ──────────────────────────────────────────────────────────

criterion = nn.CrossEntropyLoss(ignore_index=0)
results = {}

print("\n" + "="*58)
print(f"  {'Model':<12} {'Test PPL':>10} {'Val PPL':>10} {'#Params':>11}")
print("="*58)
for model, name in [(rnn_model,'RNN'), (gru_model,'GRU'), (lstm_model,'LSTM')]:
    test_loss = evaluate(model, test_loader, criterion)
    val_loss  = evaluate(model, val_loader,  criterion)
    test_ppl, val_ppl = compute_perplexity(test_loss), compute_perplexity(val_loss)
    results[name] = {'test_ppl': test_ppl, 'val_ppl': val_ppl, 'params': count_params(model)}
    print(f"  {name:<12} {test_ppl:>10.1f} {val_ppl:>10.1f} {count_params(model):>11,}")
print("="*58)
print("  ↑ Lower perplexity = better language model")


In [ ]:
# ── Learning Curves ──────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Training Dynamics on WikiText-2', fontsize=13, fontweight='bold')

COLORS = {'RNN': '#e74c3c', 'LSTM': '#2980b9', 'GRU': '#27ae60'}
EPOCHS = list(range(1, N_EPOCHS+1))

for name, hist in [('RNN', rnn_hist), ('LSTM', lstm_hist), ('GRU', gru_hist)]:
    c = COLORS[name]
    axes[0].plot(EPOCHS, hist['train_ppl'], '--', color=c, alpha=0.5, label=f'{name} Train')
    axes[0].plot(EPOCHS, hist['val_ppl'],   '-',  color=c, lw=2.5, label=f'{name} Val')

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Perplexity')
axes[0].set_title('Perplexity over Epochs (↓ better)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Bar chart of test PPL
names = ['Vanilla RNN', 'GRU', 'LSTM']
ppls  = [results['RNN']['test_ppl'], results['GRU']['test_ppl'], results['LSTM']['test_ppl']]
bars  = axes[1].bar(names, ppls, color=[COLORS['RNN'], COLORS['GRU'], COLORS['LSTM']],
                    edgecolor='white', linewidth=1.5, width=0.55)
for bar, val in zip(bars, ppls):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{val:.1f}', ha='center', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Test Perplexity'); axes[1].set_title('Test Perplexity Comparison')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: learning_curves.png")


## Section 7: Autoregressive Text Generation (SLP3 §13.3.3)

Implementation of the autoregressive generation algorithm:
1. Encode seed phrase → get hidden state
2. At each step: feed token → get logits → apply temperature → sample
3. Append sampled token, repeat until `<eos>` or max_length

**Temperature** $T$ reshapes the distribution: $p_i \propto \exp(\text{logit}_i / T)$
- $T < 1$: sharper (more conservative, less surprising)
- $T > 1$: flatter (more diverse, potentially less coherent)

In [ ]:
def generate(model, seed, vocab, idx2word, max_len=60, temperature=0.8, top_k=40):
    """Autoregressive generation — SLP3 §13.3.3"""
    model.eval()
    tokens = [vocab.get(w, vocab['<unk>']) for w in seed.lower().split()] or [vocab['<sos>']]
    generated = tokens.copy()

    with torch.no_grad():
        x = torch.tensor([tokens], dtype=torch.long, device=device)
        hidden = model.init_hidden(1)
        logits, hidden = model(x, hidden)
        inp = torch.tensor([[generated[-1]]], dtype=torch.long, device=device)

        for _ in range(max_len):
            logits, hidden = model(inp, hidden)
            logits = logits[:, -1, :] / temperature
            if top_k > 0:                                # top-k nucleus filtering
                top_vals, _ = torch.topk(logits, top_k)
                logits[logits < top_vals[:, -1:]] = float('-inf')
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            generated.append(nxt)
            if nxt == vocab['<eos>']: break
            inp = torch.tensor([[nxt]], dtype=torch.long, device=device)

    return ' '.join(idx2word.get(t, '<unk>') for t in generated)

# ── Qualitative Comparison ────────────────────────────────────────────────────
seeds = [
    "the history of",
    "in the nineteenth century",
    "scientists have discovered",
    "the university was founded in",
]

print("\n" + "="*72)
print("  QUALITATIVE TEXT GENERATION COMPARISON (temperature=0.8, top_k=40)")
print("="*72)
for seed in seeds:
    print(f"\n📌 Seed: \"{seed}\"")
    print("-"*72)
    for m, n in [(rnn_model,'RNN '), (gru_model,'GRU '), (lstm_model,'LSTM')]:
        print(f"  [{n}] {generate(m, seed, vocab, idx2word)}")


In [ ]:
# ── Temperature Ablation Study ────────────────────────────────────────────────

seed = "the history of science"
print(f"\n🌡️  Temperature Ablation | Model: LSTM | Seed: \"{seed}\"")
print("="*72)
for T in [0.4, 0.7, 1.0, 1.4]:
    label = {0.4:'conservative', 0.7:'balanced', 1.0:'raw model', 1.4:'creative/noisy'}[T]
    gen = generate(lstm_model, seed, vocab, idx2word, temperature=T)
    print(f"  T={T} ({label}):")
    print(f"    {gen}")
    print()


## Section 8: Results Analysis & Discussion

### 8.1 Why LSTM > GRU > RNN?

The performance ranking reflects how well each architecture handles **long-range dependencies**:

| Architecture | Gradient Path | Memory Capacity | Typical Outcome |
|---|---|---|---|
| Vanilla RNN | Multiplicative (vanishes) | Hidden state only | Forgets context after ~20 tokens |
| GRU | Protected by gates | Hidden state | Good mid-range memory |
| LSTM | Cell state + gates | Cell + hidden | Best long-range memory |

### 8.2 Parameter Efficiency

GRU achieves ~85–90% of LSTM's quality with ~25% fewer parameters.  
For production systems with tight latency constraints, GRU is often preferred.

### 8.3 Qualitative Observations

| Phenomenon | RNN | GRU | LSTM |
|---|---|---|---|
| Repetition | Frequent ("the the the") | Occasional | Rare |
| Subject-verb agreement | Fails across clauses | Usually correct | Mostly correct |
| Named entity consistency | Poor | Better | Best |
| Sentence completeness | Often incomplete | Usually complete | Complete |

### 8.4 Architecture Selection Guide

```
Scenario                        Recommended Architecture
─────────────────────────────────────────────────────────
Learning / education baseline   Vanilla RNN
Speed-critical, limited memory  GRU
Best quality on sequential NLP  LSTM
Long documents, large scale     Transformer (GPT-style)
Streaming/online processing     GRU or LSTM (RNNs excel here)
```

### 8.5 Connection to Modern LLMs

Today's GPT-class models are Transformer-based, but the autoregressive generation principle (SLP3 §13.3.3) is **identical**:
- Predict next token conditioned on all previous tokens
- Sample from output distribution with temperature
- Feed sampled token back as next input

RNNs remain foundational for understanding sequence modeling before moving to attention mechanisms.

### 8.6 Further Exploration

- Increase `N_EPOCHS` to 10–15 for lower perplexity
- Try `HIDDEN_SIZE=512` for a larger model
- Implement **beam search** instead of sampling
- Add **dropout on recurrent connections** (DropConnect)
- Try **AWD-LSTM** (Merity et al., 2017) for state-of-the-art LSTM results
- Read SLP3 Ch.9–10 for Transformers as the natural successor

In [ ]:
# ── Final Summary Table ───────────────────────────────────────────────────────

import pandas as pd

df = pd.DataFrame({
    'Model':          ['Vanilla RNN', 'GRU', 'LSTM'],
    'Parameters':     [f"{results['RNN']['params']/1e6:.2f}M",
                       f"{results['GRU']['params']/1e6:.2f}M",
                       f"{results['LSTM']['params']/1e6:.2f}M"],
    'Val PPL':        [f"{results['RNN']['val_ppl']:.1f}",
                       f"{results['GRU']['val_ppl']:.1f}",
                       f"{results['LSTM']['val_ppl']:.1f}"],
    'Test PPL':       [f"{results['RNN']['test_ppl']:.1f}",
                       f"{results['GRU']['test_ppl']:.1f}",
                       f"{results['LSTM']['test_ppl']:.1f}"],
    'Gating':         ['None (tanh)', '2 gates', '3 gates + cell'],
    'Best use case':  ['Education/baseline', 'Speed+quality', 'Best NLP quality'],
})

print("\n" + "="*85)
print("  FINAL COMPARISON SUMMARY — WikiText-2")
print("="*85)
print(df.to_string(index=False))
print("="*85)
print()
print("📖 Theory: https://web.stanford.edu/~jurafsky/slp3/13.pdf (Chapter 13)")
print("🔗 Repo:   https://github.com/YOUR_USERNAME/autoregressive-lm-tutorial")
print()
print("✅ Tutorial complete! Well done, Suraj & Debajyoti 🎉")
